# Salinity preprocessing notebook

Workflow:

1. Configure paths, filters, and one baseline file
2. Convert `.xyz` → COG in `cogs_staging/salinity/` (**same filename**, `.tif` instead of `.xyz`)
3. Compute salinity increase in `cogs_staging/salinity_increase/` (same filenames)
4. **Rename** COGs using the naming rule `{scenario}_{probability}_{year}.tif`
5. **Organize** renamed COGs into `stac_folder/` (same level as `cogs_staging`)


In [ ]:
from pathlib import Path
import re
import shutil

import numpy as np
import pandas as pd
import rasterio
from rasterio.shutil import copy as rio_copy
from rasterio.transform import from_origin


## 1) Configure paths and options

Define the single baseline file with `baseline_scenario` and `baseline_year`.


In [38]:
source_dir = Path(r"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\salinity_mekong\projections_gridded")
output_root = Path(r"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\salinity_mekong\preprocessed_outputs")

probability = "p50"
allowed_suffixes = ["y", "sb2y", "sm2y", "sb2rb1y", "sm2rb1y", "sb2rb3y"]

# Single baseline file (must exist after conversion)
baseline_scenario = "cc45y"
baseline_year = "2018"

resolution_m = 2000
source_crs = "EPSG:32648"
nodata_value = -9999

config = {
    "source_dir": source_dir,
    "output_root": output_root,
    "probability": probability,
    "allowed_suffixes": allowed_suffixes,
    "baseline_scenario": baseline_scenario,
    "baseline_year": baseline_year,
    "resolution_m": resolution_m,
    "source_crs": source_crs,
    "nodata_value": nodata_value,
}

# Path variables only (folders are created when needed)
cogs_staging = output_root / "cogs_staging"
salinity_staging = cogs_staging / "salinity"
salinity_increase_staging = cogs_staging / "salinity_increase"

# Final STAC output (sibling folder of cogs_staging)
stac_folder = output_root / "stac_folder"

print("Source:", source_dir)
print("Output:", output_root)
print("COG staging:", cogs_staging)
print("STAC folder:", stac_folder)
print("Expected baseline xyz stem:", f"*_{baseline_scenario}{baseline_year[-2:]}.xyz")


Source: N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\salinity_mekong\projections_gridded
Output: N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\salinity_mekong\preprocessed_outputs
COG staging: N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\salinity_mekong\preprocessed_outputs\cogs_staging
STAC folder: N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\salinity_mekong\preprocessed_outputs\stac_folder
Expected baseline xyz stem: *_cc45y18.xyz


## 2) Helper functions


In [39]:
SHORT_NAME_RE = re.compile(r"^(p\d+)_(cc\d{2}[a-z0-9]+)(\d{2})\.(xyz|tif)$", re.IGNORECASE)
XYZ_FILTER_RE = re.compile(r"_(cc\d{2}[a-z0-9]+?)(\d+)\.(xyz|tif)$", re.IGNORECASE)
RENAMED_NAME_RE = re.compile(r"^(cc\d{2}[a-z0-9]+)_(p\d+)_(\d{4})\.tif$", re.IGNORECASE)


def parse_tif_metadata(file_path):
    """Read scenario, probability and year from xyz-style or renamed filenames."""
    name = file_path.name

    renamed = RENAMED_NAME_RE.match(name)
    if renamed:
        scenario = renamed.group(1).lower()
        return {
            "scenario": scenario,
            "probability": renamed.group(2).lower(),
            "year": renamed.group(3),
            "suffix": scenario[4:],
        }

    short = SHORT_NAME_RE.match(name)
    if short:
        scenario = short.group(2).lower()
        return {
            "scenario": scenario,
            "probability": short.group(1).lower(),
            "year": f"20{short.group(3)}",
            "suffix": scenario[4:],
        }

    match = XYZ_FILTER_RE.search(name.lower())
    if match:
        scenario = match.group(1).lower()
        return {
            "scenario": scenario,
            "probability": name.lower().split("_")[0],
            "year": f"20{match.group(2)}",
            "suffix": scenario[4:],
        }

    return None


def matches_criteria(file_path, config):
    """Return file metadata only when probability and scenario suffix match config."""
    parsed = parse_tif_metadata(file_path)
    if not parsed:
        return None
    if parsed["probability"] != config["probability"].lower():
        return None
    if parsed["suffix"] not in config["allowed_suffixes"]:
        return None
    return parsed


def convert_xyz_to_cog(xyz_path, out_tif_path, config):
    """Convert one xyz file to COG."""
    out_tif_path.parent.mkdir(parents=True, exist_ok=True)

    resolution_m = config["resolution_m"]
    source_crs = config["source_crs"]
    nodata_value = config["nodata_value"]

    df = pd.read_csv(xyz_path, sep=r"\s+", header=None, names=["x", "y", "z"])
    df_raster = df.dropna(subset=["z"]).copy()
    if df_raster.empty:
        raise ValueError(f"No valid z values found in {xyz_path.name}")

    xmin, ymin = df_raster[["x", "y"]].min()
    xmax, ymax = df_raster[["x", "y"]].max()
    xmin -= resolution_m / 2
    ymin -= resolution_m / 2
    xmax += resolution_m / 2
    ymax += resolution_m / 2

    width = int(np.ceil((xmax - xmin) / resolution_m))
    height = int(np.ceil((ymax - ymin) / resolution_m))
    ymax_aligned = ymin + height * resolution_m
    transform = from_origin(xmin, ymax_aligned, resolution_m, resolution_m)

    raster = np.full((height, width), nodata_value, dtype="float32")
    cols = ((df_raster["x"].to_numpy() - xmin) // resolution_m).astype(int)
    rows = ((ymax_aligned - df_raster["y"].to_numpy()) // resolution_m).astype(int)
    vals = df_raster["z"].to_numpy(dtype="float32")
    valid = (rows >= 0) & (rows < height) & (cols >= 0) & (cols < width)
    raster[rows[valid], cols[valid]] = vals[valid]

    temp_tif = out_tif_path.with_name(out_tif_path.stem + "_temp.tif")
    with rasterio.open(
        temp_tif,
        "w",
        driver="GTiff",
        height=height,
        width=width,
        count=1,
        dtype=raster.dtype,
        crs=source_crs,
        transform=transform,
        nodata=nodata_value,
        compress="LZW",
    ) as dst:
        dst.write(raster, 1)

    rio_copy(temp_tif, out_tif_path, copy_src_overviews=True, driver="COG", compress="LZW")
    temp_tif.unlink(missing_ok=True)


def compute_salinity_increase(salinity_folder, increase_folder, config):
    """Subtract one baseline COG from all non-baseline salinity COGs."""
    salinity_folder.mkdir(parents=True, exist_ok=True)
    increase_folder.mkdir(parents=True, exist_ok=True)

    baseline_tif = None
    for tif in salinity_folder.glob("*.tif"):
        parsed = matches_criteria(tif, config)
        if not parsed:
            continue
        if parsed["scenario"] == config["baseline_scenario"] and parsed["year"] == config["baseline_year"]:
            baseline_tif = tif
            break

    if baseline_tif is None:
        raise FileNotFoundError(
            f"Baseline COG not found in {salinity_folder}. "
            f"Expected scenario={config['baseline_scenario']} and year={config['baseline_year']}."
        )

    with rasterio.open(baseline_tif) as src:
        baseline_data = src.read(1)

    rows = []
    errors = []

    for tif in sorted(salinity_folder.glob("*.tif")):
        parsed = matches_criteria(tif, config)
        if not parsed or parsed["year"] == config["baseline_year"]:
            continue

        out_tif = increase_folder / tif.name
        try:
            with rasterio.open(tif) as src:
                data = src.read(1)
                profile = src.profile.copy()

            with rasterio.open(out_tif, "w", **profile) as dst:
                dst.write(data - baseline_data, 1)

            rows.append({"input": str(tif), "baseline": str(baseline_tif), "output": str(out_tif)})
        except Exception as exc:
            errors.append({"file": str(tif), "error": str(exc)})

    return rows, errors


def rename(folder, config):
    """Rename xyz-style COGs to {scenario}_{probability}_{year}.tif."""
    folder.mkdir(parents=True, exist_ok=True)
    rows = []
    errors = []

    for tif in sorted(folder.glob("*.tif")):
        if RENAMED_NAME_RE.match(tif.name):
            continue

        parsed = matches_criteria(tif, config)
        if not parsed:
            continue

        new_name = f"{parsed['scenario']}_{parsed['probability']}_{parsed['year']}.tif"
        new_path = folder / new_name

        try:
            if new_path.exists() and new_path.resolve() != tif.resolve():
                tif.unlink()
            elif new_path != tif:
                tif.rename(new_path)
            rows.append({"old_name": tif.name, "new_name": new_name, "scenario": parsed["scenario"], "year": parsed["year"]})
        except Exception as exc:
            errors.append({"file": str(tif), "error": str(exc)})

    return rows, errors


def organize(folder, stac_folder, config, product="salinity"):
    """Move renamed COGs into STAC folders as {probability}_{year}.tif."""
    folder.mkdir(parents=True, exist_ok=True)
    rows = []
    errors = []

    for tif in sorted(folder.glob("*.tif")):
        parsed = matches_criteria(tif, config)
        if not parsed:
            continue

        scenario = parsed["scenario"]
        probability = parsed["probability"]
        year = parsed["year"]

        if not RENAMED_NAME_RE.match(tif.name):
            renamed_path = folder / f"{scenario}_{probability}_{year}.tif"
            if renamed_path.exists():
                continue

        final_name = f"{probability}_{year}.tif"

        if product == "salinity":
            if year == config["baseline_year"]:
                dst_folder = stac_folder / "salinity" / "baseline"
            else:
                dst_folder = stac_folder / "salinity" / scenario
        else:
            dst_folder = stac_folder / "salinity_increase" / scenario

        dst_folder.mkdir(parents=True, exist_ok=True)
        dst = dst_folder / final_name

        try:
            shutil.copy2(tif, dst)
            rows.append({"source": str(tif), "scenario": scenario, "year": year, "destination": str(dst)})
        except Exception as exc:
            errors.append({"file": str(tif), "error": str(exc)})

    return rows, errors


## 3) Convert selected XYZ files to salinity COGs

COGs keep the **same name as the source xyz** (only extension changes).


In [40]:
conversion_rows = []
conversion_errors = []
skipped_files = []

for xyz_path in sorted(config["source_dir"].glob("*.xyz")):
    parsed = matches_criteria(xyz_path, config)
    if not parsed:
        skipped_files.append(str(xyz_path))
        continue

    out_tif = salinity_staging / xyz_path.with_suffix(".tif").name

    try:
        convert_xyz_to_cog(xyz_path, out_tif, config)
        conversion_rows.append({
            "xyz": str(xyz_path),
            "scenario": parsed["scenario"],
            "year": parsed["year"],
            "cog": str(out_tif),
        })
    except Exception as exc:
        conversion_errors.append({"xyz": str(xyz_path), "error": str(exc)})

print(f"Selected and converted: {len(conversion_rows)}")
print(f"Skipped (did not match criteria): {len(skipped_files)}")
print(f"Conversion errors: {len(conversion_errors)}")

if conversion_rows:
    display(pd.DataFrame(conversion_rows).sort_values(["scenario", "year"]))
if conversion_errors:
    display(pd.DataFrame(conversion_errors))


Selected and converted: 16
Skipped (did not match criteria): 15
Conversion errors: 0


,xyz,scenario,year,cog
0,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...",cc45sm2rb1y,2030,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."
1,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...",cc45sm2rb1y,2040,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."
2,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...",cc45sm2rb1y,2050,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."
3,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...",cc45sm2y,2030,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."
4,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...",cc45sm2y,2040,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."
5,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...",cc45sm2y,2050,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."
6,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...",cc45y,2018,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."
7,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...",cc45y,2030,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."
8,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...",cc45y,2040,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."
9,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...",cc45y,2050,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."


## 4) Compute salinity increase

Uses the single baseline defined in section 1. Stops with a clear error if baseline is missing.


In [41]:
increase_rows, increase_errors = compute_salinity_increase(
    salinity_staging,
    salinity_increase_staging,
    config,
)

print(f"Salinity increase files created: {len(increase_rows)}")
print(f"Errors: {len(increase_errors)}")

if increase_rows:
    display(pd.DataFrame(increase_rows))
if increase_errors:
    display(pd.DataFrame(increase_errors))


Salinity increase files created: 15
Errors: 0


,input,baseline,output
0,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...","N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...","N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."
1,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...","N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...","N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."
2,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...","N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...","N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."
3,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...","N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...","N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."
4,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...","N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...","N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."
5,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...","N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...","N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."
6,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...","N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...","N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."
7,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...","N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...","N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."
8,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...","N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...","N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."
9,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...","N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...","N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."


## 5) Rename COGs in staging folders

Renaming rule:

`{scenario}_{probability}_{year}.tif`

This step runs **before** organization.


In [42]:
salinity_rename_rows, salinity_rename_errors = rename(salinity_staging, config)
increase_rename_rows, increase_rename_errors = rename(salinity_increase_staging, config)

print(f"Salinity renamed: {len(salinity_rename_rows)}")
print(f"Salinity increase renamed: {len(increase_rename_rows)}")

if salinity_rename_rows:
    display(pd.DataFrame(salinity_rename_rows))
if increase_rename_rows:
    display(pd.DataFrame(increase_rename_rows))

all_rename_errors = salinity_rename_errors + increase_rename_errors
if all_rename_errors:
    display(pd.DataFrame(all_rename_errors))


Salinity renamed: 16
Salinity increase renamed: 15


,old_name,new_name,scenario,year
0,P50_cc45sm2rb1y30.tif,cc45sm2rb1y_p50_2030.tif,cc45sm2rb1y,2030
1,P50_cc45sm2rb1y40.tif,cc45sm2rb1y_p50_2040.tif,cc45sm2rb1y,2040
2,P50_cc45sm2rb1y50.tif,cc45sm2rb1y_p50_2050.tif,cc45sm2rb1y,2050
3,P50_cc45sm2y30.tif,cc45sm2y_p50_2030.tif,cc45sm2y,2030
4,P50_cc45sm2y40.tif,cc45sm2y_p50_2040.tif,cc45sm2y,2040
5,P50_cc45sm2y50.tif,cc45sm2y_p50_2050.tif,cc45sm2y,2050
6,P50_cc45y18.tif,cc45y_p50_2018.tif,cc45y,2018
7,P50_cc45y30.tif,cc45y_p50_2030.tif,cc45y,2030
8,P50_cc45y40.tif,cc45y_p50_2040.tif,cc45y,2040
9,P50_cc45y50.tif,cc45y_p50_2050.tif,cc45y,2050


,old_name,new_name,scenario,year
0,P50_cc45sm2rb1y30.tif,cc45sm2rb1y_p50_2030.tif,cc45sm2rb1y,2030
1,P50_cc45sm2rb1y40.tif,cc45sm2rb1y_p50_2040.tif,cc45sm2rb1y,2040
2,P50_cc45sm2rb1y50.tif,cc45sm2rb1y_p50_2050.tif,cc45sm2rb1y,2050
3,P50_cc45sm2y30.tif,cc45sm2y_p50_2030.tif,cc45sm2y,2030
4,P50_cc45sm2y40.tif,cc45sm2y_p50_2040.tif,cc45sm2y,2040
5,P50_cc45sm2y50.tif,cc45sm2y_p50_2050.tif,cc45sm2y,2050
6,P50_cc45y30.tif,cc45y_p50_2030.tif,cc45y,2030
7,P50_cc45y40.tif,cc45y_p50_2040.tif,cc45y,2040
8,P50_cc45y50.tif,cc45y_p50_2050.tif,cc45y,2050
9,P50_cc85sb2y30.tif,cc85sb2y_p50_2030.tif,cc85sb2y,2030


## 6) Organize into final folder structure

Final filename in STAC folders:

`{probability}_{year}.tif`

Files are saved in `stac_folder/` (same level as `cogs_staging`):

- `stac_folder/salinity/baseline/`
- `stac_folder/salinity/{scenario}/`
- `stac_folder/salinity_increase/{scenario}/`


In [43]:
salinity_org_rows, salinity_org_errors = organize(salinity_staging, stac_folder, config, product="salinity")
increase_org_rows, increase_org_errors = organize(salinity_increase_staging, stac_folder, config, product="salinity_increase")

print(f"Salinity organized: {len(salinity_org_rows)}")
print(f"Salinity increase organized: {len(increase_org_rows)}")

if salinity_org_rows:
    display(pd.DataFrame(salinity_org_rows).sort_values(["scenario", "year"]))
if increase_org_rows:
    display(pd.DataFrame(increase_org_rows).sort_values(["scenario", "year"]))

all_org_errors = salinity_org_errors + increase_org_errors
if all_org_errors:
    display(pd.DataFrame(all_org_errors))


Salinity organized: 16
Salinity increase organized: 15


,source,scenario,year,destination
0,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...",cc45sm2rb1y,2030,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."
1,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...",cc45sm2rb1y,2040,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."
2,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...",cc45sm2rb1y,2050,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."
3,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...",cc45sm2y,2030,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."
4,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...",cc45sm2y,2040,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."
5,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...",cc45sm2y,2050,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."
6,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...",cc45y,2018,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."
7,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...",cc45y,2030,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."
8,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...",cc45y,2040,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."
9,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...",cc45y,2050,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."


,source,scenario,year,destination
0,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...",cc45sm2rb1y,2030,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."
1,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...",cc45sm2rb1y,2040,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."
2,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...",cc45sm2rb1y,2050,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."
3,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...",cc45sm2y,2030,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."
4,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...",cc45sm2y,2040,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."
5,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...",cc45sm2y,2050,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."
6,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...",cc45y,2030,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."
7,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...",cc45y,2040,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."
8,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...",cc45y,2050,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."
9,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s...",cc85sb2y,2030,"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\s..."
